<a href="https://colab.research.google.com/github/helonayala/sysid/blob/main/lagrange_quarter_drone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lagrangian mechanics

This script demonstrates a symbolic workflow applies modeling and linearizing a mechanical system, with an application to the 1/4 drone test bench.



## Imports and functions

In [1]:
!pip install control
import sympy as sp
import numpy as np
import control as ct
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.integrate import solve_ivp # Import the standard ODE solver

# Use SymPy's best available renderer for equations
sp.init_printing(use_latex='mathjax')

# --- Helper Functions for Generalization ---

def create_gen_vars(ndof, t):
    """Creates symbolic generalized coordinates, velocities, and accelerations."""
    q = sp.Matrix([sp.Function(f'q{i+1}')(t) for i in range(ndof)])
    qp = q.diff(t)
    qpp = qp.diff(t)
    return q, qp, qpp

def derive_eom(T, V, q, Fext, t):
    """Derives the symbolic EOM using the Euler-Lagrange equation."""
    ndof = len(q)
    L = T - V
    qp = q.diff(t)
    qpp = qp.diff(t)
    EOM_LHS = sp.Matrix([
        (sp.diff(sp.diff(L, qp[i]), t) - sp.diff(L, q[i])).simplify()
        for i in range(ndof)
    ])
    return sp.Eq(EOM_LHS, Fext), qpp


def get_state_space_representation(EOM, q, qp, qpp, u_vec):
    """Converts the symbolic EOM into a symbolic state-space representation."""
    ndof = len(q)
    sol = sp.solve(EOM, qpp, dict=True)[0]
    q_s = sp.symbols(f'q1:{ndof+1}_s')
    qp_s = sp.symbols(f'q1:{ndof+1}p_s')

    sub_map = {}
    for i in range(ndof):
        sub_map[q[i]] = q_s[i]
        sub_map[qp[i]] = qp_s[i]
        sub_map[sp.sin(q[i])] = sp.sin(q_s[i])
        sub_map[sp.cos(q[i])] = sp.cos(q_s[i])

    x = sp.Matrix([*q_s, *qp_s])
    u = u_vec

    xp_list = list(qp_s)
    for i in range(ndof):
        xp_list.append(sol[qpp[i]].subs(sub_map))
    xp = sp.Matrix(xp_list)

    return xp, x, u



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 578.3/578.3 kB 8.8 MB/s eta 0:00:00


## Energy terms

We use for this example the inverted pendulum on a cart. Only spung and unsprung masses are considered for inertia, and damping is present in $\theta$ angle of rotation (0 is pointing upwards).

In [2]:
# =============================================================================
# 1. System Definition
# =============================================================================
print("--- 1. System Definition ---")

# Define symbolic variables for the system parameters
# m1: mass with motor, m2: counter-mass, L = rod lenght, equal for both mass, g = acceleration due to gravity, t = time, b' = viscous friction
m1, m2, L, g, t, b, k_tau = sp.symbols('m1 m2 L g t b k_tau')

# Define symbolic variable for the input torque, function based on input tau_in = C*u, where C is a constant. As seen in the experiments, it is necessary two for each sign of u_in (piecewise function)
u_in = sp.symbols('u_in')

# Define generalized coordinates, velocities, and accelerations
ndof = 1  # One degree of freedom (rotation)
print(f"\nNumber of Degrees of Freedom (ndof): {ndof}")

q, qp, qpp = create_gen_vars(ndof, t)
print("\nGeneralized coordinates vector (q):")
display(q)


# =============================================================================
# 2. Kinematics, Energy, and Forces for the Quarter Drone
# =============================================================================
print("\n--- 2. Kinematics, Energy, and Forces (from X,Y coordinates) ---")

# Unpack the generalized coordinate (theta) and velocity for easier use
q1 = q[0]
q1p = qp[0]

# --- Kinematics (Positions) ---
# Define the X,Y position of each mass based on the angle q1 (theta).
# q1 = 0 corresponds to the rod pointing vertically upwards.
print("\nDefining X,Y positions:")
x1 = L * sp.sin(q1)
y1 = L * sp.cos(q1)
x2 = -L * sp.sin(q1)
y2 = -L * sp.cos(q1)
print("Position of mass m1: (x1, y1)")
display(sp.Matrix([x1, y1]))
print("Position of mass m2: (x2, y2)")
display(sp.Matrix([x2, y2]))

# --- Potential Energy (V) ---
# V is calculated from the height (Y coordinate) of each mass.
# The reference V=0 is at the pivot (y=0).
V = m1 * g * y1 + m2 * g * y2
print("\nPotential Energy (V = m1*g*y1 + m2*g*y2):")
display(V)

# --- Kinetic Energy (T) ---
# T is calculated from the X and Y velocities of each mass.
# v^2 = (dx/dt)^2 + (dy/dt)^2
v1_sq = sp.diff(x1, t)**2 + sp.diff(y1, t)**2
v2_sq = sp.diff(x2, t)**2 + sp.diff(y2, t)**2
print("\nSquared velocity of mass m1 (v1_sq):")
display(v1_sq)

# The total kinetic energy is the sum for both masses.
# We use sp.simplify() to resolve terms like sin(q1)**2 + cos(q1)**2 = 1.
T = sp.simplify(0.5 * m1 * v1_sq + 0.5 * m2 * v2_sq)
print("\nTotal Kinetic Energy (T = 0.5*m1*v1_sq + 0.5*m2*v2_sq):")
display(T)


# --- Generalized External Forces (Fext) ---
# This term is unchanged. It includes input torque and rotational friction.
Fext = sp.Matrix([k_tau * u_in - b * q1p])
print("\nGeneralized external forces vector (Fext):")
display(Fext)

# --- Control Input Vector (u) ---
u_vec = sp.Matrix([u_in])

--- 1. System Definition ---

Number of Degrees of Freedom (ndof): 1

Generalized coordinates vector (q):


[q₁(t)]


--- 2. Kinematics, Energy, and Forces (from X,Y coordinates) ---

Defining X,Y positions:
Position of mass m1: (x1, y1)


⎡L⋅sin(q₁(t))⎤
⎢            ⎥
⎣L⋅cos(q₁(t))⎦

Position of mass m2: (x2, y2)


...

                          2                             2
 2    2        ⎛d        ⎞     2    2        ⎛d        ⎞ 
L ⋅sin (q₁(t))⋅⎜──(q₁(t))⎟  + L ⋅cos (q₁(t))⋅⎜──(q₁(t))⎟ 
               ⎝dt       ⎠                   ⎝dt       ⎠ 


Total Kinetic Energy (T = 0.5*m1*v1_sq + 0.5*m2*v2_sq):


                            2
     2           ⎛d        ⎞ 
0.5⋅L ⋅(m₁ + m₂)⋅⎜──(q₁(t))⎟ 
                 ⎝dt       ⎠ 


Generalized external forces vector (Fext):


⎡    d                   ⎤
⎢- b⋅──(q₁(t)) + kₜₐᵤ⋅uᵢₙ⎥
⎣    dt                  ⎦

## Obtaining state equations

In [3]:
# =============================================================================
# 3. Automated EOM Derivation
# =============================================================================
print("--- 3. Automated EOM Derivation ---")
EOM, qpp = derive_eom(T, V, q, Fext, t)

# Display the symbolic Equations of Motion
print("\nDerived Equations of Motion (EOM):")
display(EOM)

# =============================================================================
# 4. Automated State-Space Formulation
# =============================================================================
print("\n--- 4. Automated State-Space Formulation ---")
xp, x, u = get_state_space_representation(EOM, q, qp, qpp, u_vec)

# Display the symbolic state-space vectors
print("\nState Vector (x):")
display(x)
print("\nInput Vector (u):")
display(u)
print("\nState Derivative Vector (xp):")
display(xp)


--- 3. Automated EOM Derivation ---

Derived Equations of Motion (EOM):


⎡  ⎛                 2                                            ⎞⎤           ↪
⎢  ⎜                d                                             ⎟⎥   ⎡    d  ↪
⎢L⋅⎜1.0⋅L⋅(m₁ + m₂)⋅───(q₁(t)) - g⋅m₁⋅sin(q₁(t)) + g⋅m₂⋅sin(q₁(t))⎟⎥ = ⎢- b⋅── ↪
⎢  ⎜                  2                                           ⎟⎥   ⎣    dt ↪
⎣  ⎝                dt                                            ⎠⎦           ↪

↪                    
↪                   ⎤
↪ (q₁(t)) + kₜₐᵤ⋅uᵢₙ⎥
↪                   ⎦
↪                    


--- 4. Automated State-Space Formulation ---

State Vector (x):


⎡q₁ ₛ⎤
⎢    ⎥
⎣q1pₛ⎦


Input Vector (u):


[uᵢₙ]


State Derivative Vector (xp):


⎡                               q1pₛ                                ⎤
⎢                                                                   ⎥
⎢L⋅g⋅m₁⋅sin(q₁ ₛ)   L⋅g⋅m₂⋅sin(q₁ ₛ)      b⋅q1pₛ         kₜₐᵤ⋅uᵢₙ   ⎥
⎢──────────────── - ──────────────── - ───────────── + ─────────────⎥
⎢  2       2          2       2         2       2       2       2   ⎥
⎣ L ⋅m₁ + L ⋅m₂      L ⋅m₁ + L ⋅m₂     L ⋅m₁ + L ⋅m₂   L ⋅m₁ + L ⋅m₂⎦

In [6]:
# =============================================================================
# 5. System Linearization
# =============================================================================
print("--- 5. System Linearization ---")
# Define equilibrium point and numerical parameters
equilibrium_point = {s: 0 for s in x}
equilibrium_point.update({s: 0 for s in u})
param_values = {m1: 0.0990, m2: 0.1306, L: 0.2723, g: 9.81, b: 0.116, k_tau: 0.1} # !!! identificar estes parametros (usando dados)

# Linearize the system
A_sym = xp.jacobian(x)
B_sym = xp.jacobian(u)
A_lin_sym = A_sym.subs(equilibrium_point)
B_lin_sym = B_sym.subs(equilibrium_point)

Alin = np.array(A_lin_sym.evalf(subs=param_values, chop=True)).astype(float)
Blin = np.array(B_lin_sym.evalf(subs=param_values, chop=True)).astype(float)

# =============================================================================
# 6. System Simplification (if there are pole/zero cancelations)
# =============================================================================
print("\n--- 6. System Simplification ---")
# Define the system output (pendulum angle)
output_index = 1
Clin = np.array([[1, 0]]) # !!! confirmar qual é a saída correspondente
Clin[0, output_index] = 1.0
Dlin = np.array([[0]])

# Get the minimal realization to find the core 2nd-order dynamics.
sys_full_ss = ct.ss(Alin, Blin, Clin, Dlin)

print("\nMinimal System Transfer Function:")
sys_min_tf = ct.minreal(ct.tf(sys_full_ss))
print(sys_min_tf)

sys_min_ss = ct.ss(sys_min_tf)
A_min, B_min, C_min, D_min = sys_min_ss.A, sys_min_ss.B, sys_min_ss.C, sys_min_ss.D


--- 5. System Linearization ---

--- 6. System Simplification ---

Minimal System Transfer Function:
0 states have been removed from the model
<TransferFunction>: sys[10]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']

     5.874 s + 5.874
  ---------------------
  s^2 + 6.814 s + 4.958
